# Week 5 Lab — Pretraining, GPT-style Models, and In-Context Learning

**Goals**
- Train a tiny GPT-style decoder-only language model.
- Observe in-context learning behavior.
- Explore evaluation pitfalls for language models.

This lab complements Week 5 of the reader.


## 0. Setup
We build a *very small* decoder-only Transformer and train it on a toy corpus.
The objective is architectural and behavioral understanding, not performance.


In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 5
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. Tiny corpus

We intentionally use a small, structured corpus so you can see memorization,
generalization, and failure modes clearly.


In [ ]:
corpus = [
    "the system shall record events",
    "the system shall verify inputs",
    "the model predicts the next token",
    "verification requires external checks",
    "optimization affects generalization",
]

text = '\n'.join(corpus)
text

## 2. Character-level tokenization

Using character-level tokens avoids tokenizer complexity and keeps focus on
model behavior.


In [ ]:
chars = sorted(list(set(text)))
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
V = len(chars)
V, chars

## 3. Dataset: next-character prediction


In [ ]:
class CharDataset(Dataset):
    def __init__(self, text, block_size):
        self.data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]
        return x, y

block_size = 32
dataset = CharDataset(text, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

len(dataset)

## 4. Minimal GPT-style model

Decoder-only Transformer with:
- token embeddings
- positional embeddings
- causal self-attention
- linear language modeling head


## 5. Training loop

Train for a small number of epochs and observe loss curves.


## 6. In-context learning probe

Prompt the trained model with patterns it has seen and slight variations.
Observe where generalization breaks.


## 7. Reflection

Answer briefly:
- Where did the model memorize vs generalize?
- What looked like in-context learning?
- How would you *mis-evaluate* this model if you were careless?
